In [ ]:
"""
Author: Sophie A. Liu
Date: 06/12/2026 10:58am
Purpose: isolating local expression activity around each immunofluorescent labeled cell
"""

In [1]:
# importing necessary libraries
import pandas as pd
import numpy as np

In [ ]:
1.1
#V = pd.read_csv("iso_sig.csv")
#V = pd.read_csv("pd1_sig.csv")
#V = pd.read_csv("iso_all_genes.csv")
#V = pd.read_csv("pd1_all_genes.csv")
#V =  pd.read_csv("mark_known.csv")

1.2
#V = pd.read_csv("9NMF_isoC.csv")
V = pd.read_csv("9NMF_Apd1.csv")

1.3
#V = pd.read_csv("9NSF_isoC.csv")
#V = pd.read_csv("9NSF_Apd1.csv")


#S = pd.read_csv("iso_coords.csv")       # spots from IF
S = pd.read_csv("apd1_coords.csv")

In [ ]:
# setting parameters/ initializing things. all the different iterations of block 1
v_coords = V[["x", "y"]].to_numpy()
s_coords = S[["x", "y"]].to_numpy()

# cell_cols = V.columns[-11:]              # 11 signatures, 25 annotated. replace factor_cols instances below with cell_cols
factor_cols = V.columns[-9:]               # 9 NMF and NSF factors

radius = 40                                # tradeoff 10 to 80 specificity & sparsity

In [9]:
from scipy.spatial import cKDTree

In [10]:
def inputs(S, V, factor_cols, s_coords, v_coords):

    # KD-trees
    s_tree = cKDTree(s_coords)
    v_tree = cKDTree(v_coords)

    # encoding our four imaging channels as integers leads to faster processing
    type_map = {
        "tdtomato": 0,
        "gc3ai": 1,
        "cd8": 2,
        "lectin": 3
    }
    S_cells = np.array([type_map.get(x, -1) for x in S["cell_type"].values])

    # extracting factor matrix :)
    V_factors = V[factor_cols].to_numpy()

    return s_tree, v_tree, S_cells, V_factors

In [ ]:
# counts of each cell type in the neighborhood of a IF-labeled cell, as well as some derived metrics.
def counts_in_radius(center, s_tree, S_cells, radius):

    idx = s_tree.query_ball_point(center, r=radius)

    if len(idx) == 0:
        counts = np.zeros(4)   # for all four types, if nothing then set 0. Loops through all neighborhoods
    else:
        types = S_cells[idx]
        counts = np.bincount(types[types >= 0], minlength=4)   # our result: counts = [n_tdtomato, n_gc3ai, n_cd8, n_lectin]

    n_alive, n_dying, n_immune, n_endothelial = counts         # renaming the channels to what cell type they represent

    # calculating later metrics so I don't have to do it downstream
    n_tumor = n_alive + n_dying
    total = n_tumor + n_immune + n_endothelial
    exist_dying = 1 if n_dying > 0 else 0

    return counts, n_tumor, total, exist_dying

In [ ]:
# mean factor expression in neighborhood, a better representation than sum bc density effects
# possible future direction is gaussian decay from center
def get_factor_means(center, v_tree, V_factors, radius):
    idx = v_tree.query_ball_point(center, r=radius)

    if len(idx) == 0:
        return np.zeros(V_factors.shape[1])

    return V_factors[idx].mean(axis=0)

In [27]:
def append_row(center, s_tree, v_tree, S_types, V_factors, radius):

    counts, n_tumor, total, exist_dying = counts_in_radius(
        center, s_tree, S_types, radius
    )

    factor_means = get_factor_means(
        center, v_tree, V_factors, radius
    )

    row = np.concatenate([
        np.array([center[0], center[1]]),
        counts,
        np.array([n_tumor, total, exist_dying]),
        factor_means
    ])

    return row

In [50]:
# function for final assembly. didn't vectorize because not that slow and this was more intuitive. 
def compute_neighborhoods(
    S, V, s_coords, v_coords, factor_cols, radius):

    s_tree, v_tree, S_types, V_factors = inputs(
        S, V, factor_cols, s_coords, v_coords
    )

    n_centers = len(s_coords)
    n_factors = V_factors.shape[1]

    results = np.zeros((n_centers, 9 + n_factors))

    for i, center in enumerate(s_coords):
        results[i] = append_row(
            center, s_tree, v_tree, S_types, V_factors, radius
        )

    columns = (
        ["x", "y",
         "n_alive", "n_dying", "n_immune", "n_lectin",
         "n_tumor", "all", "exist_dying"]
        + list(factor_cols)
    )

    return pd.DataFrame(results, columns=columns)

In [59]:
df = compute_neighborhoods(
    S=S,
    V=V,
    s_coords=s_coords,
    v_coords=v_coords,
    factor_cols=factor_cols,
    radius=radius
)

df = df.join(S[["cell_type", "sample"]])               # reattaching metadata

In [60]:
df_clean = df[df["n_tumor"] > 0]

In [53]:
# helps counter dependence by sampling non-overlapping neighborhoods using a greedy algorithm.
# random hard-core thinning. Matern soft potentially better but this made the most sense to me
def non_overlapping(df, n, radius, seed):
    coords = df[['x', 'y']].to_numpy()
    remaining_idx = np.arange(len(coords))
    rng = np.random.default_rng(seed=42)               # the answer to life, the universe, and everything

    selected_idx = []

    while len(selected_idx) < n and len(remaining_idx) > 0:
        pick_i = rng.choice(remaining_idx)
        selected_idx.append(pick_i)

        tree = cKDTree(coords[remaining_idx])

        # getting rid of all other points in that radius
        neighbors = tree.query_ball_point(coords[pick_i], r=radius)
        to_remove = set(remaining_idx[neighbors])

        remaining_idx = np.array([i for i in remaining_idx if i not in to_remove])

    return df.iloc[selected_idx].copy()

In [61]:
df_sub = non_overlapping(df_clean, n = 1000,            # for comparability. rule of thumb 383 so we good
                             radius=radius*2)   

In [63]:
#df_sub.to_csv("sig_pd140.csv", index=False)
#df_sub.to_csv("all_genes_pd140.csv", index=False)
#df_sub.to_csv("all_genes_pd140.csv", index=False)
#df_sub.to_csv("DE_pd140.csv", index=False)

df_sub.to_csv("9NMF_pd140.csv", index=False)
#df_sub.to_csv("9NMF_pd110.csv", index=False)           # both conds. varying radii inspect
#df_sub.to_csv("9NMF_pd120.csv", index=False)
#df_sub.to_csv("9NMF_pd180.csv", index=False)
#df_sub.to_csv("9NSF_pd140.csv", index=False)